# Python Dependency Checker

주어진 파일/모듈/함수를 **import하거나 호출하는** Python 파일을 프로젝트 내에서 검색합니다.

MATLAB의 `checkDependencyInSubfolders.m`과 동일한 목적이지만:
- AST 기반 import 분석 (정적 의존성)
- 텍스트 기반 문자열 검색 (동적 호출, 주석 포함)
- 두 결과를 통합해 표시

## 사용법
1. **Section 1** 파라미터 설정
2. **Section 2** 전체 실행 (`Run All`)
3. **Section 3** 결과 확인

## Section 1 — 검색 파라미터

In [ ]:
from pathlib import Path

# ─── 검색 루트 디렉토리 ─────────────────────────────────────────────────────
# None이면 이 노트북 파일 기준 eMach 루트를 자동 감지
SEARCH_ROOT: str | None = None

# ─── 검색 대상 (하나 이상 설정) ──────────────────────────────────────────────
# 예: 파일명          "doe_batch"          → doe_batch.py를 import/from하는 파일
# 예: 모듈 경로        "pyMCAD.doe_batch"   → from pyMCAD.doe_batch import ... 포함
# 예: 함수명           "doe_batch_run"      → 함수 호출/import 포함
# 예: 클래스명         "DOEPoint"           → 클래스 사용 포함
TARGET = "doe_batch"

# ─── 검색 범위 옵션 ──────────────────────────────────────────────────────────
SEARCH_IMPORTS  = True    # AST 기반: import / from ... import 구문 분석
SEARCH_TEXT     = True    # 텍스트 기반: 파일 내 단순 문자열 포함 여부
EXCLUDE_COMMENTS = False  # True이면 # 주석 라인 제외 (텍스트 검색에만 적용)
CASE_SENSITIVE  = True    # False이면 대소문자 무시

# ─── 제외 패턴 ───────────────────────────────────────────────────────────────
EXCLUDE_DIRS  = {'.git', '__pycache__', '.ipynb_checkpoints', 'node_modules', '.venv', 'venv'}
EXCLUDE_FILES = set()     # 예: {'dependency_checker.ipynb'}

# ─── 파일 확장자 ─────────────────────────────────────────────────────────────
FILE_EXTS = {'.py', '.ipynb'}   # .ipynb 포함 시 셀 소스를 텍스트로 검색

# ─── 결과 출력 옵션 ──────────────────────────────────────────────────────────
SHOW_MATCHED_LINES = True   # 매칭된 줄 내용 표시
MAX_LINES_PER_FILE = 5      # 파일당 최대 표시 줄 수 (0이면 전체)

## Section 2 — 의존성 체커 유틸리티 함수

In [ ]:
import ast
import json
import os
import re
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional, Set


@dataclass
class MatchResult:
    """단일 파일에서 발견된 의존성 결과."""
    file_path: Path
    match_type: str          # 'import' | 'text'
    matched_lines: List[tuple]  # [(line_no, line_content), ...]
    import_names: List[str] = field(default_factory=list)

    @property
    def rel_path(self) -> Path:
        """search_root 기준 상대 경로."""
        try:
            return self.file_path.relative_to(_SEARCH_ROOT)
        except ValueError:
            return self.file_path


# ─── AST import 분석 ──────────────────────────────────────────────────────────

def _extract_imports_from_source(source: str) -> List[tuple]:
    """소스 코드에서 (module_or_name, lineno) 목록 추출."""
    results = []
    try:
        tree = ast.parse(source)
    except SyntaxError:
        return results

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                results.append((alias.name, node.lineno))
        elif isinstance(node, ast.ImportFrom):
            mod = node.module or ""
            results.append((mod, node.lineno))
            for alias in node.names:
                full = f"{mod}.{alias.name}" if mod else alias.name
                results.append((full, node.lineno))
    return results


def _source_imports_target(source: str, target: str, case_sensitive: bool) -> List[tuple]:
    """소스가 target 모듈/이름을 import하면 매칭된 (lineno, snippet) 반환."""
    imports = _extract_imports_from_source(source)
    t = target if case_sensitive else target.lower()
    matched = []
    seen_lines: Set[int] = set()
    for (name, lineno) in imports:
        n = name if case_sensitive else name.lower()
        # name이 target과 동일하거나, target을 포함하거나, target의 부모 모듈인 경우
        if t in n or n == t or t.split('.')[-1] == n.split('.')[-1]:
            if lineno not in seen_lines:
                seen_lines.add(lineno)
                lines = source.splitlines()
                snippet = lines[lineno - 1].strip() if lineno <= len(lines) else ""
                matched.append((lineno, snippet))
    return matched


# ─── 텍스트 검색 ─────────────────────────────────────────────────────────────

def _text_search_in_source(source: str, target: str, case_sensitive: bool,
                            exclude_comments: bool) -> List[tuple]:
    """소스에서 target 문자열이 포함된 (lineno, line) 목록 반환."""
    matched = []
    t = target if case_sensitive else target.lower()
    for i, line in enumerate(source.splitlines(), start=1):
        l = line if case_sensitive else line.lower()
        if exclude_comments:
            stripped = line.lstrip()
            if stripped.startswith('#'):
                continue
        if t in l:
            matched.append((i, line.rstrip()))
    return matched


# ─── Jupyter notebook 소스 추출 ──────────────────────────────────────────────

def _get_notebook_source(file_path: Path) -> str:
    """노트북의 모든 code/markdown 셀을 하나의 문자열로 합침."""
    try:
        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            nb = json.load(f)
        lines = []
        for cell in nb.get('cells', []):
            src = cell.get('source', [])
            if isinstance(src, list):
                lines.append(''.join(src))
            else:
                lines.append(str(src))
        return '\n'.join(lines)
    except Exception:
        return ''


# ─── 단일 파일 검사 ───────────────────────────────────────────────────────────

def check_file(
    file_path: Path,
    target: str,
    *,
    search_imports: bool = True,
    search_text: bool = True,
    case_sensitive: bool = True,
    exclude_comments: bool = False,
) -> Optional[MatchResult]:
    """파일 하나를 검사해 매칭되면 MatchResult, 없으면 None 반환."""
    ext = file_path.suffix.lower()

    if ext == '.ipynb':
        source = _get_notebook_source(file_path)
        # 노트북은 텍스트 검색만 (AST는 셀 번호 매핑이 불명확)
        if not search_text:
            return None
        text_matches = _text_search_in_source(source, target, case_sensitive, exclude_comments)
        if text_matches:
            return MatchResult(
                file_path=file_path,
                match_type='text',
                matched_lines=text_matches,
            )
        return None

    # .py 파일
    try:
        source = file_path.read_text(encoding='utf-8', errors='replace')
    except Exception:
        return None

    import_matches: List[tuple] = []
    import_names: List[str] = []
    if search_imports:
        import_matches = _source_imports_target(source, target, case_sensitive)

    text_matches: List[tuple] = []
    if search_text:
        text_matches = _text_search_in_source(source, target, case_sensitive, exclude_comments)
        # import 매칭과 중복 라인 제거 (import 라인은 이미 import_matches에 있음)
        import_linenos = {ln for ln, _ in import_matches}
        text_matches = [(ln, l) for ln, l in text_matches if ln not in import_linenos]

    if not import_matches and not text_matches:
        return None

    match_type = 'import+text' if (import_matches and text_matches) else ('import' if import_matches else 'text')
    all_matches = sorted(set(import_matches + text_matches), key=lambda x: x[0])

    return MatchResult(
        file_path=file_path,
        match_type=match_type,
        matched_lines=all_matches,
        import_names=import_names,
    )


# ─── 전체 디렉토리 스캔 ───────────────────────────────────────────────────────

def scan_dependencies(
    target: str,
    search_root: str | Path,
    *,
    file_exts: Set[str] = {'.py', '.ipynb'},
    exclude_dirs: Set[str] = EXCLUDE_DIRS,
    exclude_files: Set[str] = EXCLUDE_FILES,
    search_imports: bool = True,
    search_text: bool = True,
    case_sensitive: bool = True,
    exclude_comments: bool = False,
    verbose: bool = True,
) -> List[MatchResult]:
    """search_root 하위의 모든 파일에서 target 의존성을 검색.

    Returns:
        target을 참조하는 파일들의 MatchResult 리스트
    """
    global _SEARCH_ROOT
    _SEARCH_ROOT = Path(search_root)

    results: List[MatchResult] = []
    n_scanned = 0

    for dirpath, dirnames, filenames in os.walk(_SEARCH_ROOT):
        # 제외 디렉토리 프루닝
        dirnames[:] = [
            d for d in dirnames
            if d not in exclude_dirs and not d.startswith('.')
        ]

        for fname in filenames:
            if fname in exclude_files:
                continue
            ext = Path(fname).suffix.lower()
            if ext not in file_exts:
                continue

            fpath = Path(dirpath) / fname
            n_scanned += 1
            result = check_file(
                fpath, target,
                search_imports=search_imports,
                search_text=search_text,
                case_sensitive=case_sensitive,
                exclude_comments=exclude_comments,
            )
            if result is not None:
                results.append(result)

    if verbose:
        print(f"Scanned {n_scanned} files under: {_SEARCH_ROOT}")
        print(f"Found {len(results)} file(s) referencing '{target}'")

    return results


# ─── 결과 출력 ────────────────────────────────────────────────────────────────

def print_results(
    results: List[MatchResult],
    target: str,
    *,
    show_matched_lines: bool = True,
    max_lines_per_file: int = 5,
) -> None:
    """검색 결과를 가독성 있게 출력."""
    if not results:
        print(f"  ('{target}'를 참조하는 파일 없음)")
        return

    print(f"\n{'═'*70}")
    print(f"  Target: '{target}'  →  {len(results)}개 파일")
    print(f"{'═'*70}")

    for r in sorted(results, key=lambda x: str(x.rel_path)):
        tag = {
            'import':      '[import      ]',
            'text':        '[text        ]',
            'import+text': '[import+text ]',
        }.get(r.match_type, f'[{r.match_type:12s}]')
        print(f"\n  {tag}  {r.rel_path}")

        if show_matched_lines and r.matched_lines:
            lines_to_show = r.matched_lines
            truncated = False
            if max_lines_per_file > 0 and len(lines_to_show) > max_lines_per_file:
                lines_to_show = lines_to_show[:max_lines_per_file]
                truncated = True
            for lineno, content in lines_to_show:
                print(f"    {lineno:>5}: {content}")
            if truncated:
                print(f"    ... (+{len(r.matched_lines) - max_lines_per_file} more lines)")

    print(f"\n{'─'*70}")


print("유틸리티 함수 로드 완료.")

## Section 3 — 실행

In [ ]:
# search root 자동 감지: 이 노트북 위치에서 eMach 루트 탐색
_nb_dir = Path().resolve()  # 노트북 실행 디렉토리

if SEARCH_ROOT is not None:
    _root = Path(SEARCH_ROOT)
else:
    # eMach 루트를 찾기: __init__.py가 없는 첫 번째 상위 폴더
    _candidate = _nb_dir
    for _p in [_nb_dir] + list(_nb_dir.parents):
        if (_p / 'eMach').exists() or (_p / 'tools').exists():
            _root = _p
            break
    else:
        _root = _nb_dir
    print(f"Auto-detected search root: {_root}")

# 자기 자신(이 노트북)은 제외
_self_name = Path('dependency_checker.ipynb').name
_exclude = EXCLUDE_FILES | {_self_name}

results = scan_dependencies(
    TARGET,
    _root,
    file_exts=FILE_EXTS,
    exclude_dirs=EXCLUDE_DIRS,
    exclude_files=_exclude,
    search_imports=SEARCH_IMPORTS,
    search_text=SEARCH_TEXT,
    case_sensitive=CASE_SENSITIVE,
    exclude_comments=EXCLUDE_COMMENTS,
    verbose=True,
)

print_results(
    results,
    TARGET,
    show_matched_lines=SHOW_MATCHED_LINES,
    max_lines_per_file=MAX_LINES_PER_FILE,
)

## Section 4 — 고급: 여러 타겟 동시 검색

In [ ]:
# 여러 모듈의 의존성을 한번에 확인하는 예시
# 각 타겟에 대해 결과 수만 요약

MULTI_TARGETS = [
    "doe_batch",
    "doe_batch_run",
    "magnetic_h5",
    "build_train_manifest",
]

print(f"{'Target':<30} {'Files':>5}  Paths")
print('─' * 70)
for t in MULTI_TARGETS:
    _r = scan_dependencies(t, _root, exclude_files=_exclude, verbose=False)
    paths = ', '.join(str(x.rel_path) for x in sorted(_r, key=lambda x: str(x.rel_path))[:3])
    ellipsis = ' ...' if len(_r) > 3 else ''
    print(f"  {t:<28} {len(_r):>5}  {paths}{ellipsis}")

## Section 5 — 역방향: 이 파일이 의존하는 모듈 목록

In [ ]:
def list_imports_of_file(file_path: str | Path) -> List[str]:
    """주어진 .py 파일이 import하는 모든 모듈/이름 목록 반환 (AST 기반)."""
    p = Path(file_path)
    if not p.exists():
        print(f"파일 없음: {p}")
        return []
    source = p.read_text(encoding='utf-8', errors='replace')
    imports = _extract_imports_from_source(source)
    unique = sorted(set(name for name, _ in imports))
    return unique


# 예시: doe_batch.py가 의존하는 모듈 목록
_target_file = _root / 'eMach' / 'tools' / 'motorCAD' / 'pyMCAD' / 'doe_batch.py'
if _target_file.exists():
    deps = list_imports_of_file(_target_file)
    print(f"{_target_file.name} 의 imports ({len(deps)}개):")
    for d in deps:
        print(f"  {d}")
else:
    print(f"파일을 찾을 수 없습니다: {_target_file}")
    print("SEARCH_ROOT 또는 경로를 직접 지정해주세요.")